In [2]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import os

os.makedirs("results", exist_ok=True)

TRADING_DAYS = 252
RISK_FREE = 0.02
ASSETS = ["EQ", "BOND", "BTC"]
CRYPTO_IDX = ASSETS.index("BTC")
CRYPTO_CAP = 0.20



def annualized_inputs(ret_slice):
    """Expected return vector and covariance matrix, annualized."""
    mu = ret_slice[ASSETS].mean().values * TRADING_DAYS
    cov = ret_slice[ASSETS].cov().values * TRADING_DAYS
    return mu, cov


def port_stats(w, mu, cov):
    r = w @ mu
    v = np.sqrt(w @ cov @ w)
    return r, v, (r - RISK_FREE) / v



def _base_constraints():
    return [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]


def _bounds(crypto_cap=CRYPTO_CAP):
    b = []
    for i in range(len(ASSETS)):
        hi = crypto_cap if i == CRYPTO_IDX else 1.0
        b.append((0.0, hi))
    return b


def max_sharpe(mu, cov, crypto_cap=CRYPTO_CAP):
    n = len(mu)
    w0 = np.repeat(1.0 / n, n)
    res = minimize(
        lambda w: -((w @ mu - RISK_FREE) / np.sqrt(w @ cov @ w)),
        w0, method="SLSQP",
        bounds=_bounds(crypto_cap), constraints=_base_constraints(),
    )
    return res.x


def min_variance_for_target(mu, cov, target_return, crypto_cap=CRYPTO_CAP):
    n = len(mu)
    w0 = np.repeat(1.0 / n, n)
    cons = _base_constraints() + [
        {"type": "ineq", "fun": lambda w: w @ mu - target_return}
    ]
    res = minimize(
        lambda w: w @ cov @ w,
        w0, method="SLSQP",
        bounds=_bounds(crypto_cap), constraints=cons,
    )
    return res.x



def sensitivity_to_btc_return(mu, cov, bumps=np.linspace(-0.15, 0.15, 13)):
    """Re-run max-Sharpe with BTC's expected return shifted by each bump
    (in absolute annualized terms, e.g. -0.15 = 15 pp lower)."""
    rows = []
    for b in bumps:
        mu_b = mu.copy()
        mu_b[CRYPTO_IDX] += b
        w = max_sharpe(mu_b, cov)
        rows.append({"btc_mu_bump": b, "btc_mu": mu_b[CRYPTO_IDX],
                     "opt_crypto_weight": w[CRYPTO_IDX]})
    return pd.DataFrame(rows)


def sensitivity_to_correlation(mu, cov, scales=np.linspace(0.25, 1.75, 13)):
    """Scale the BTC-vs-others correlations by each factor (variances
    unchanged) and re-run max-Sharpe. scale > 1 = more correlated."""
    sd = np.sqrt(np.diag(cov))
    corr = cov / np.outer(sd, sd)
    rows = []
    for s in scales:
        c = corr.copy()
        for i in range(len(ASSETS)):
            if i != CRYPTO_IDX:
                c[i, CRYPTO_IDX] = np.clip(corr[i, CRYPTO_IDX] * s, -0.99, 0.99)
                c[CRYPTO_IDX, i] = c[i, CRYPTO_IDX]
        cov_s = c * np.outer(sd, sd)
        w = max_sharpe(mu, cov_s)
        rows.append({"corr_scale": s, "opt_crypto_weight": w[CRYPTO_IDX]})
    return pd.DataFrame(rows)



def risk_parity(cov):
    """Equal risk contribution portfolio (long-only, fully invested).
    Minimizes squared deviations of each asset's risk contribution from
    the average contribution."""
    n = cov.shape[0]

    def objective(w):
        port_var = w @ cov @ w
        mrc = cov @ w
        rc = w * mrc
        return np.sum((rc - port_var / n) ** 2)

    w0 = np.repeat(1.0 / n, n)
    res = minimize(objective, w0, method="SLSQP",
                   bounds=[(0.001, 1.0)] * n,
                   constraints=_base_constraints())
    return res.x


def utility_allocation(mu, cov, lam, crypto_cap=CRYPTO_CAP):
    """max  w'mu - (lam/2) w'cov w   subject to long-only, sum=1, cap."""
    n = len(mu)
    w0 = np.repeat(1.0 / n, n)
    res = minimize(
        lambda w: -(w @ mu - 0.5 * lam * (w @ cov @ w)),
        w0, method="SLSQP",
        bounds=_bounds(crypto_cap), constraints=_base_constraints(),
    )
    return res.x



def run_optimization(logret, periods, target_return=0.08):
    all_rows = []
    sens_mu_frames, sens_corr_frames, util_frames = {}, {}, {}

    for pname, (s, e) in periods.items():
        ret_slice = logret.loc[s:e]
        mu, cov = annualized_inputs(ret_slice)


        w_ms = max_sharpe(mu, cov)
        w_mv = min_variance_for_target(mu, cov, target_return)


        w_rp = risk_parity(cov)
        lams = [1, 2, 4, 8, 16]
        w_ut = {lam: utility_allocation(mu, cov, lam) for lam in lams}

        for label, w in [("Max Sharpe", w_ms),
                         (f"Min Var (target {target_return:.0%})", w_mv),
                         ("Risk Parity", w_rp)] + \
                        [(f"Utility (lambda={lam})", w_ut[lam]) for lam in lams]:
            r, v, sh = port_stats(w, mu, cov)
            row = {"period": pname, "method": label,
                   "w_EQ": w[0], "w_BOND": w[1], "w_BTC": w[2],
                   "exp_return": r, "exp_vol": v, "sharpe": sh}
            all_rows.append(row)


        sens_mu_frames[pname] = sensitivity_to_btc_return(mu, cov)
        sens_corr_frames[pname] = sensitivity_to_correlation(mu, cov)
        util_frames[pname] = pd.DataFrame(
            {"lambda": lams, "w_BTC": [w_ut[l][CRYPTO_IDX] for l in lams]})

    summary = pd.DataFrame(all_rows)
    summary.to_csv("results/optimization_summary.csv", index=False)
    return summary, sens_mu_frames, sens_corr_frames, util_frames



def plot_optimal_weights(summary):
    methods = ["Max Sharpe", "Risk Parity"]
    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=160)
    periods_list = summary["period"].unique()
    x = np.arange(len(periods_list))
    w = 0.36
    colors = ["#3E6C8C", "#B5473A"]
    for i, m in enumerate(methods):
        vals = [summary[(summary.period == p) & (summary.method == m)]["w_BTC"].iloc[0]
                for p in periods_list]
        bars = ax.bar(x + (i - 0.5) * w, vals, w, color=colors[i], label=m)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.004, f"{v:.1%}",
                    ha="center", fontsize=10, fontweight="bold")
    ax.axhline(CRYPTO_CAP, color="black", linestyle="--", linewidth=1,
               label=f"Crypto cap ({CRYPTO_CAP:.0%})")
    ax.set_xticks(x); ax.set_xticklabels(periods_list)
    ax.set_ylabel("Optimal BTC weight")
    ax.set_title("Optimizer's Recommended Crypto Weight by Regime",
                 fontweight="bold")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.savefig("results/opt_weights_by_regime.png"); plt.close()


def plot_sensitivity(sens_mu_frames, sens_corr_frames):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=160)
    colors = {"calm": "#3E6C8C", "stress": "#B5473A"}
    for pname, df in sens_mu_frames.items():
        axes[0].plot(df["btc_mu_bump"] * 100, df["opt_crypto_weight"] * 100,
                     "-o", markersize=4, label=pname,
                     color=colors.get(pname, None))
    axes[0].set_xlabel("Shift in assumed BTC expected return (pp, annualized)")
    axes[0].set_ylabel("Optimal BTC weight (%)")
    axes[0].set_title("Sensitivity to Expected-Return Assumption", fontweight="bold")
    axes[0].legend(frameon=False)

    for pname, df in sens_corr_frames.items():
        axes[1].plot(df["corr_scale"], df["opt_crypto_weight"] * 100,
                     "-o", markersize=4, label=pname,
                     color=colors.get(pname, None))
    axes[1].set_xlabel("Scale applied to BTC-equity/bond correlation")
    axes[1].set_ylabel("Optimal BTC weight (%)")
    axes[1].set_title("Sensitivity to Correlation Assumption", fontweight="bold")
    axes[1].legend(frameon=False)

    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(linestyle="--", alpha=0.3)
    plt.tight_layout(); plt.savefig("results/opt_sensitivity.png"); plt.close()


def plot_utility_curve(util_frames):
    fig, ax = plt.subplots(figsize=(7, 4.4), dpi=160)
    colors = {"calm": "#3E6C8C", "stress": "#B5473A"}
    for pname, df in util_frames.items():
        ax.plot(df["lambda"], df["w_BTC"] * 100, "-o",
                label=pname, color=colors.get(pname, None))
    ax.set_xscale("log", base=2)
    ax.set_xticks([1, 2, 4, 8, 16]); ax.set_xticklabels([1, 2, 4, 8, 16])
    ax.set_xlabel("Risk-aversion parameter (lambda)")
    ax.set_ylabel("BTC weight (%)")
    ax.set_title("Utility-Based Allocation vs. Risk Aversion", fontweight="bold")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(linestyle="--", alpha=0.3)
    plt.tight_layout(); plt.savefig("results/opt_utility_curve.png"); plt.close()



if __name__ == "__main__":

    rng = np.random.default_rng(7)
    n = 2000
    dates = pd.bdate_range("2018-01-01", periods=n)


    eq = rng.normal(0.0004, 0.010, n)
    bond = rng.normal(0.00012, 0.004, n)
    btc = np.empty(n)
    half = n // 2
    btc[:half] = 0.4 * eq[:half] / 0.010 * 0.035 + rng.normal(0.002, 0.030, half)
    btc[half:] = 0.85 * eq[half:] / 0.010 * 0.045 + rng.normal(-0.002, 0.042, n - half)

    logret = pd.DataFrame({"EQ": eq, "BOND": bond, "BTC": btc}, index=dates)
    periods = {
        "calm": (str(dates[0].date()), str(dates[half - 1].date())),
        "stress": (str(dates[half].date()), str(dates[-1].date())),
    }

    summary, sens_mu, sens_corr, util = run_optimization(logret, periods)

    pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
    print("\n=== Optimization Summary ===")
    print(summary.to_string(index=False))

    plot_optimal_weights(summary)
    plot_sensitivity(sens_mu, sens_corr)
    plot_utility_curve(util)
    print("\nCharts + CSV written to ./results/")


=== Optimization Summary ===
period              method  w_EQ  w_BOND  w_BTC  exp_return  exp_vol  sharpe
  calm          Max Sharpe 0.000   0.800  0.200       0.053    0.119   0.277
  calm Min Var (target 8%) 0.000   0.739  0.200       0.052    0.117   0.273
  calm         Risk Parity 0.452   0.458  0.090      -0.011    0.103  -0.305
  calm  Utility (lambda=1) 0.000   0.800  0.200       0.053    0.119   0.277
  calm  Utility (lambda=2) 0.000   0.800  0.200       0.053    0.119   0.277
  calm  Utility (lambda=4) 0.000   0.825  0.175       0.048    0.108   0.263
  calm  Utility (lambda=8) 0.000   0.908  0.092       0.033    0.077   0.170
  calm Utility (lambda=16) 0.000   0.949  0.051       0.026    0.067   0.082
stress          Max Sharpe 0.214   0.786  0.000       0.060    0.060   0.664
stress Min Var (target 8%) 0.943   0.057  0.000       0.080    0.154   0.391
stress         Risk Parity 0.473   0.489  0.038       0.062    0.108   0.387
stress  Utility (lambda=1) 1.000   0.000  0.00